# 03 - Analysis
Cross-correlation, Granger causality, event detection, and impact assessment.

In [ ]:
import sys
sys.path.insert(0, "..")
import logging
logging.basicConfig(level=logging.INFO)
import pandas as pd
import numpy as np

## Step 1: Load data

In [ ]:
from src.polymarket.market_discovery import load_discovered_markets
from src.polymarket.timeseries import TimeseriesFetcher
from src.freight.scraper import fetch_all_freight_indexes
from src.freight.normalize import prepare_freight_panel

markets_df = load_discovered_markets()
fetcher = TimeseriesFetcher()
timeseries = fetcher.fetch_all(markets_df)
freight_raw = fetch_all_freight_indexes(use_synthetic_fallback=True)
freight_data = prepare_freight_panel(freight_raw)

print(f'Markets: {len(timeseries)}, Freight indexes: {list(freight_data.keys())}')

## Step 2: Event Detection

In [ ]:
from src.analysis.events import EventDetector

detector = EventDetector()
all_events = detector.detect_all(timeseries, markets_df)
events_df = detector.to_dataframe(all_events)
print(f'Detected {len(all_events)} significant probability shift events')
events_df.head(20)

In [ ]:
# Top events by magnitude
top_events = detector.get_top_events(all_events, n=10)
top_events_df = detector.to_dataframe(top_events)
print('Top 10 most significant probability shifts:')
top_events_df[['market_title', 'timestamp', 'probability_before',
               'probability_after', 'delta', 'magnitude', 'direction']]

## Step 3: Cross-Correlation Analysis

In [ ]:
from src.analysis.correlation import CorrelationAnalyser

analyser = CorrelationAnalyser()
xcorr_results = analyser.run_cross_correlations(timeseries, freight_data, markets_df)
xcorr_df = analyser.xcorr_to_dataframe(xcorr_results)
print(f'Cross-correlation computed for {len(xcorr_results)} market-freight pairings')
xcorr_df.sort_values('peak_correlation', ascending=False)

In [ ]:
# Show significant results
sig_df = xcorr_df[xcorr_df['is_significant']]
print(f'
Statistically significant pairings (p < 0.05): {len(sig_df)}')
print(f"Pairings where Polymarket leads freight: {sig_df['polymarket_leads'].sum()}")
sig_df[['market_title', 'freight_index', 'peak_lag_days',
        'peak_correlation', 'peak_p_value', 'interpretation']]

## Step 4: Granger Causality

In [ ]:
granger_results = analyser.run_granger_tests(timeseries, freight_data, markets_df)
granger_df = analyser.granger_to_dataframe(granger_results)
print(f'Granger tests run for {len(granger_results)} pairings')
print(f"Significant (p < 0.05): {granger_df['is_significant'].sum()}")
granger_df.sort_values('min_p_value')

## Step 5: Impact Assessment

In [ ]:
from src.analysis.impact_mapper import ImpactMapper

mapper = ImpactMapper()
assessments = mapper.generate_assessments(all_events, markets_df, xcorr_results)
assessments_df = mapper.to_dataframe(assessments)
print(f'Generated {len(assessments)} impact assessments')
assessments_df[['signal', 'timestamp', 'confidence', 'impact_score', 'category']].head(10)

In [ ]:
# Print the top 3 assessments in full
for a in assessments[:3]:
    print('=' * 70)
    print(f'SIGNAL: {a.signal}')
    print(f'Date: {a.timestamp} | Category: {a.category} | Confidence: {a.confidence.upper()}')
    print(f'Impact Score: {a.impact_score:.4f}')
    print(f'
Affected Routes: {
.join(a.affected_routes)}')
    print(f'Predicted Impact: {a.predicted_impact}')
    print(f'Expected Range: {a.predicted_impact_range}')
    print(f'
Recommended Actions:')
    for action in a.recommended_actions[:4]:
        print(f'  - {action}')
    print(f'
Historical Precedent: {a.historical_precedent}')
    print()

## Summary
Analysis complete. Proceed to 04_whitepaper_figures.ipynb for publication-quality charts.